In [1]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.colors as mcolors

from mpl_toolkits.axes_grid1 import make_axes_locatable


# -------------------------
# PDF compatibility
# -------------------------
# Prevent some PDF readers from reporting:
# "Insufficient data for an image"
mpl.rcParams["pdf.compression"] = 0



In [2]:

# -------------------------
# Style
# -------------------------
sns.set_theme(style="white", context="paper")

plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "font.size": 11,
    "axes.labelsize": 12,
    "axes.titlesize": 13,
    "axes.linewidth": 0.8,
    "font.family": "Times New Roman",
})

LINE_COLOR = "#2C3E50"


# -------------------------
# Project paths
# -------------------------
# The Jupyter Notebook working directory should be:
# Code/Figures Python
project_dir = Path.cwd()

data_dir = project_dir / "Data"
results_dir = project_dir / "Results"


# -------------------------
# Input and output paths
# -------------------------
file_xlsx = (
    data_dir
    / "MSA_PFF_GO_Analyses_top300.xlsx"
)

out_dir = (
    results_dir
    / "MSA_PFF_GO_Analyses_top300"
    / "GO_Analyses"
)

out_dir.mkdir(
    parents=True,
    exist_ok=True
)


# -------------------------
# Sheets
# -------------------------
sheets = [
    "MSA_BP",
    "PFF_BP",
    "MSA_MF",
    "PFF_MF",
    "MSA_CC",
    "PFF_CC",
]


# -------------------------
# Group-wise FE-axis unification
# Only MSA and PFF plots from the same ontology
# share the same Fold Enrichment axis.
# -------------------------
groups = {
    "BP": [
        "MSA_BP",
        "PFF_BP"
    ],
    "MF": [
        "MSA_MF",
        "PFF_MF"
    ],
    "CC": [
        "MSA_CC",
        "PFF_CC"
    ],
}


# -------------------------
# Display names for ontologies
# -------------------------
ONTOLOGY_LABELS = {
    "BP": "Biological process",
    "MF": "Molecular function",
    "CC": "Cellular component",
}


# -------------------------
# Column names and settings
# -------------------------
COL_TERM = "GO Term"
COL_FE = "Fold Enrichment"
COL_P = "Bonferroni P-value"

TOP_N = 30
P_FLOOR = 1e-300


# -------------------------
# Deep, low-brightness blue-to-red colormap
# -------------------------
deep_blue_red = mcolors.LinearSegmentedColormap.from_list(
    "deep_blue_red",
    [
        (0.08, 0.18, 0.32),  # deep navy
        (0.18, 0.32, 0.52),
        (0.40, 0.40, 0.45),  # muted middle
        (0.55, 0.28, 0.30),
        (0.45, 0.05, 0.08),  # deep wine red
    ],
    N=256
)

cmap = deep_blue_red


# -------------------------
# Read one sheet
# -------------------------
def read_sheet(sheet):
    """
    Read one GO enrichment sheet and retain the required columns.

    Color values are calculated as:

        -log10(Bonferroni p) + 1
    """
    df = pd.read_excel(
        file_xlsx,
        sheet_name=sheet
    )

    # Remove possible spaces from column names.
    df.columns = (
        df.columns
        .astype(str)
        .str.strip()
    )

    df = df[
        [
            COL_TERM,
            COL_FE,
            COL_P
        ]
    ].copy()

    df[COL_FE] = pd.to_numeric(
        df[COL_FE],
        errors="coerce"
    )

    df[COL_P] = pd.to_numeric(
        df[COL_P],
        errors="coerce"
    ).clip(
        lower=P_FLOOR
    )

    df = df.dropna(
        subset=[
            COL_TERM,
            COL_FE,
            COL_P
        ]
    )

    # MATLAB-style transformation
    df["mlog10p_plus1"] = (
        -np.log10(
            df[COL_P].to_numpy(dtype=float)
        )
        + 1.0
    )

    return df


# -------------------------
# Read all sheets
# -------------------------
all_data = {
    sheet: read_sheet(sheet)
    for sheet in sheets
}


# -------------------------
# Global unified color scale
# -------------------------
# Use one color scale across all six figures.
all_vals = np.concatenate([
    all_data[sheet]["mlog10p_plus1"].to_numpy(dtype=float)
    for sheet in sheets
])

# Use a robust range so extreme values do not dominate the colormap.
vmin = float(
    np.nanpercentile(
        all_vals,
        5
    )
)

vmax = float(
    np.nanpercentile(
        all_vals,
        95
    )
)

if vmax <= vmin:
    vmax = vmin + 1e-6

norm = mpl.colors.Normalize(
    vmin=vmin,
    vmax=vmax
)


# -------------------------
# Unify FE axis within each ontology
# -------------------------
fe_xlim_by_group = {}

for group_name, sheet_list in groups.items():
    fe_max = max(
        all_data[sheet][COL_FE].max()
        for sheet in sheet_list
    )

    fe_xlim_by_group[group_name] = (
        0.0,
        float(fe_max) * 1.05
    )


# -------------------------
# Identify ontology from sheet name
# -------------------------
def group_name_from_sheet(sheet):
    """
    Return BP, MF, or CC according to the sheet suffix.
    """
    if sheet.endswith("_BP"):
        return "BP"

    if sheet.endswith("_MF"):
        return "MF"

    if sheet.endswith("_CC"):
        return "CC"

    return None


# -------------------------
# Identify dataset from sheet name
# -------------------------
def dataset_name_from_sheet(sheet):
    """
    Return MSA or PFF according to the sheet prefix.
    """
    if sheet.startswith("MSA_"):
        return "MSA"

    if sheet.startswith("PFF_"):
        return "PFF"

    return sheet.split("_")[0]


# -------------------------
# Build publication-style figure title
# -------------------------
def title_from_sheet(sheet):
    """
    Convert a sheet name such as MSA_BP into:

        MSA: Biological process
    """
    ontology_code = group_name_from_sheet(sheet)
    dataset_name = dataset_name_from_sheet(sheet)

    ontology_name = ONTOLOGY_LABELS.get(
        ontology_code,
        ontology_code
    )

    return f"{dataset_name}: {ontology_name}"


# -------------------------
# Plot
# -------------------------
def plot_go_bar(sheet, df):
    """
    Plot the top GO terms ranked by Bonferroni p value.

    Bar length:
        Fold Enrichment

    Bar color:
        -log10(Bonferroni p) + 1
    """
    group_name = group_name_from_sheet(
        sheet
    )

    xlim = fe_xlim_by_group.get(
        group_name,
        None
    )

    # Select the TOP_N most significant GO terms.
    dfp = (
        df
        .sort_values(
            COL_P,
            ascending=True
        )
        .head(TOP_N)
        .copy()
    )

    # Reorder for horizontal display:
    # smaller FE at the bottom, larger FE at the top.
    dfp = dfp.sort_values(
        COL_FE,
        ascending=True
    )

    y = np.arange(
        len(dfp)
    )

    colors = cmap(
        norm(
            dfp["mlog10p_plus1"].to_numpy(dtype=float)
        )
    )

    # Fixed width; height depends on number of GO terms.
    FIG_WIDTH = 9.5

    fig_h = max(
        6,
        0.35 * len(dfp) + 2
    )

    fig, ax = plt.subplots(
        figsize=(
            FIG_WIDTH,
            fig_h
        ),
        constrained_layout=False
    )

    ax.barh(
        y,
        dfp[COL_FE].to_numpy(dtype=float),
        color=colors,
        edgecolor="none"
    )

    ax.set_yticks(
        y
    )

    ax.set_yticklabels(
        dfp[COL_TERM].astype(str).to_numpy(),
        fontsize=10
    )

    ax.set_xlabel(
        "Fold Enrichment"
    )

    # Use full ontology names instead of BP/MF/CC abbreviations.
    ax.set_title(
        title_from_sheet(sheet)
    )

    # Use the same FE range for MSA and PFF within each ontology.
    if xlim is not None:
        ax.set_xlim(
            *xlim
        )

    ax.tick_params(
        axis="both",
        direction="out",
        length=3,
        width=0.8
    )

    for spine in ax.spines.values():
        spine.set_linewidth(
            0.8
        )

    # Add a separate colorbar axis to the right.
    divider = make_axes_locatable(
        ax
    )

    cax = divider.append_axes(
        "right",
        size="3.5%",
        pad=0.15
    )

    sm = mpl.cm.ScalarMappable(
        norm=norm,
        cmap=cmap
    )

    sm.set_array([])

    cbar = fig.colorbar(
        sm,
        cax=cax
    )

    cbar.set_label(
        r"$-\log_{10}(\mathrm{Bonferroni}\ p) + 1$"
    )

    cbar.outline.set_linewidth(
        0.8
    )

    # Leave enough space for long GO-term labels.
    fig.subplots_adjust(
        left=0.35,
        right=0.88
    )

    pdf_path = (
        out_dir
        / f"{sheet}_GO_barplot.pdf"
    )

    png_path = (
        out_dir
        / f"{sheet}_GO_barplot.png"
    )

    fig.savefig(
        pdf_path
    )

    fig.savefig(
        png_path,
        dpi=300
    )

    plt.close(fig)

    print(
        "Saved:",
        title_from_sheet(sheet)
    )



In [3]:

# -------------------------
# Run
# -------------------------
for sheet in sheets:
    plot_go_bar(
        sheet,
        all_data[sheet]
    )

print("\nAll saved to:\n", out_dir)

Saved: MSA: Biological process
Saved: PFF: Biological process
Saved: MSA: Molecular function
Saved: PFF: Molecular function
Saved: MSA: Cellular component
Saved: PFF: Cellular component

All saved to:
 C:\Users\forge\Desktop\Documents\8. GCI vs PFF Spread model\Code\Figures Python\Results\MSA_PFF_GO_Analyses_top300\GO_Analyses
